# Dumbbell grid figure

Per-orientation grid: rows = action category, columns = prompt
(plain | evaluation-aware). Each panel is a dumbbell over [Human + 7 LMs];
the two dots are the cell's macro % in scaffolding-labelled vs
rigor-labelled moments.

Layout match to the paper version:
  - Human row sits at the top with a soft grey band and a bold tick label;
    its connector is thicker than the LM ones.
  - Marker SHAPE encodes the tutor (matches the action-distribution figure);
    marker COLOR encodes the situation (blue = scaffolding, purple = rigor).
  - Connector color encodes appropriateness: green when usage shifts toward
    the appropriate situation, red when it shifts away, grey for neutral.
  - Action-name labels on the LEFT, tutor labels on the RIGHT.

Emits three PDFs (one per orientation), matching the paper's three subfigures.
Consumes the classified facet CSVs produced by `tutorsim taxonomy classify`.
Requires `matplotlib` beyond the `taxonomy` extras.


In [ ]:
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

from tutorsim import taxonomy as tx

REPO = Path.cwd()
while not (REPO / ".git").exists() and REPO != REPO.parent:
    REPO = REPO.parent

# Update these to point at the classified.csv files produced by
#   tutorsim taxonomy classify --output <dir>
HUMAN_CLASSIFIED = REPO / "data" / "taxonomy" / "human" / "classified.csv"
LM_CLASSIFIED    = REPO / "data" / "taxonomy" / "lm"    / "classified.csv"
OUT_DIR          = REPO / "analysis" / "working-paper-20260630"


In [ ]:
# Tutor identity (shape) -- keep in sync with the action-distribution figure.
MODEL_DISPLAY = {
    "claude-opus-4-8":             "Claude Opus 4.8",
    "claude-sonnet-4-6":           "Claude Sonnet 4.6",
    "deepseek-ai/DeepSeek-V4-Pro": "DeepSeek V4 Pro",
    "gemini-2.5-pro":              "Gemini 2.5 Pro",
    "gemini-3.5-flash":            "Gemini 3.5 Flash",
    "gpt-5.5-2026-04-23":          "GPT 5.5",
    "gpt-5.4-mini-2026-03-17":     "GPT 5.4 mini",
}
MODEL_MARKERS = {
    "claude-opus-4-8":             "v",
    "claude-sonnet-4-6":           "s",
    "deepseek-ai/DeepSeek-V4-Pro": "^",
    "gemini-2.5-pro":              "D",
    "gemini-3.5-flash":            "p",
    "gpt-5.5-2026-04-23":          "P",
    "gpt-5.4-mini-2026-03-17":     "X",
}
HUMAN_MARKER = "o"  # circle reserved for the human row.
ROW_KEYS   = ["human"] + list(MODEL_DISPLAY.keys())
ROW_LABELS = ["Human tutors"] + list(MODEL_DISPLAY.values())
ROW_MARKER = {"human": HUMAN_MARKER, **MODEL_MARKERS}

# Layout colours.
SCAFF_SIT_COLOR = "#2b7bba"  # scaffolding-situation dot
RIGOR_SIT_COLOR = "#8e44ad"  # rigor-situation dot
GOOD_COLOR      = "#2e8b57"  # connector: usage in the appropriate direction
BAD_COLOR       = "#c0392b"  # connector: usage in the mismatched direction
NEUTRAL_BAR     = "#87898b"  # connector: neutral orientation
HUMAN_BAND      = "#1C2B33"  # soft tint behind the human row
HUMAN_BAND_ALPHA = 0.08
PAGE_BG         = "white"

# Short labels for the action-name column (left side). M (Other) is dropped.
SHORT_LABELS = {
    "A": "Guiding questions",
    "B": "Breaking into steps",
    "C": "Explaining / modeling",
    "D": "Alternative reps",
    "E": "Hints",
    "F": "Supplying answers",
    "G": "Prompting justification",
    "H": "Independent work",
    "I": "Increasing complexity",
    "J": "Self-assessment",
    "K": "Affirmations / check-ins",
    "L": "Transitioning",
}


In [ ]:
human = tx.read_classified_csv(HUMAN_CLASSIFIED)
lm    = tx.read_classified_csv(LM_CLASSIFIED)

human_df = tx.facets_to_dataframe(human)
lm_df    = tx.facets_to_dataframe(lm)

# Per-(situation, letter) macro % for the human pool.
h_sit = tx.macro_distribution(human_df, group_keys=("situation_label",))
# Per-(model, prompt, situation, letter) macro % for the LM pool.
lm_sit = tx.macro_distribution(
    lm_df, group_keys=("model", "prompt", "situation_label"))

def pct(row_key, prompt, situation, letter):
    if row_key == "human":
        m = h_sit[(h_sit["situation_label"] == situation)
                  & (h_sit["letter"] == letter)]
    else:
        m = lm_sit[(lm_sit["model"] == row_key)
                   & (lm_sit["prompt"] == prompt)
                   & (lm_sit["situation_label"] == situation)
                   & (lm_sit["letter"] == letter)]
    return float(m["mean_pct"].iloc[0]) if len(m) else 0.0

print("sample (human, A, plain, scaffolding-situation):", pct("human", "plain", "scaffolding", "A"))


In [ ]:
def _contained_xlim(dmin, dmax):
    pad = max(0.4, 0.08 * dmax)
    return max(0.0, dmin - pad), dmax + pad

def draw_panel(ax, letter, prompt, orientation, show_ticklabels):
    y = np.arange(len(ROW_KEYS))
    scaff = np.array([pct(r, prompt, "scaffolding", letter) for r in ROW_KEYS])
    rigor = np.array([pct(r, prompt, "rigor", letter)       for r in ROW_KEYS])
    # Soft band behind the human row (index 0).
    ax.axhspan(-0.5, 0.5, color=HUMAN_BAND, alpha=HUMAN_BAND_ALPHA, zorder=0)
    for yi, key, s, r in zip(y, ROW_KEYS, scaff, rigor):
        if orientation == "scaffolding":
            bar_c = GOOD_COLOR if s >= r else BAD_COLOR
        elif orientation == "rigor":
            bar_c = GOOD_COLOR if r >= s else BAD_COLOR
        else:
            bar_c = NEUTRAL_BAR
        is_human = (key == "human")
        lw    = 2.6 if is_human else 1.4
        alpha = 0.9 if is_human else 0.5
        marker = ROW_MARKER[key]
        ax.plot([r, s], [yi, yi], color=bar_c, lw=lw, alpha=alpha,
                solid_capstyle="round", zorder=2)
        ax.scatter(r, yi, s=50, color=RIGOR_SIT_COLOR, marker=marker,
                   edgecolor=PAGE_BG, linewidth=0.5, zorder=3)
        ax.scatter(s, yi, s=50, color=SCAFF_SIT_COLOR, marker=marker,
                   edgecolor=PAGE_BG, linewidth=0.5, zorder=3)
    ax.set_yticks(y)
    if show_ticklabels:
        ax.yaxis.tick_right()
        ax.set_yticklabels(ROW_LABELS, fontsize=10)
        ax.get_yticklabels()[0].set_fontweight("bold")
    else:
        ax.set_yticklabels([])
    ax.set_ylim(len(ROW_KEYS) - 0.5, -0.5)
    ax.grid(axis="x", ls=":", alpha=0.35)
    ax.set_axisbelow(True)


In [ ]:
def plot_orientation_grid(orientation, out_pdf):
    letters = [L for L in SHORT_LABELS
               if tx.ORIENTATION_BY_LETTER[L] == orientation]
    if not letters:
        return None
    prompts = [("plain", "Plain prompt"),
               ("scaffolding_rigor", "Evaluation-aware prompt")]
    nrow, ncol = len(letters), len(prompts)
    figh = 2.5 * nrow + 1.8
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, figh),
                              squeeze=False, sharex="row")

    for i, letter in enumerate(letters):
        for j, (prompt, plabel) in enumerate(prompts):
            ax = axes[i][j]
            draw_panel(ax, letter, prompt, orientation,
                       show_ticklabels=(j == ncol - 1))
            if i == 0:
                ax.set_title(plabel, fontsize=11, fontweight="bold", pad=10)
        # Action label on the LEFT side, perpendicular to the axis.
        axes[i][0].yaxis.set_label_position("left")
        axes[i][0].set_ylabel(
            textwrap.fill(SHORT_LABELS[letter], 18),
            rotation=0, ha="right", va="center",
            fontsize=11, fontweight="bold", labelpad=14,
        )
        # Shared x-limit per row (so both prompts align).
        row_vals = []
        for prompt, _ in prompts:
            for key in ROW_KEYS:
                row_vals.append(pct(key, prompt, "scaffolding", letter))
                row_vals.append(pct(key, prompt, "rigor", letter))
        lo, hi = _contained_xlim(min(row_vals), max(row_vals))
        for j in range(ncol):
            axes[i][j].set_xlim(lo, hi)

    for j in range(ncol):
        axes[-1][j].set_xlabel(
            "Share of tutor actions in that situation (%)", fontsize=11)

    handles = [
        Line2D([0], [0], marker="o", linestyle="none",
               markerfacecolor=SCAFF_SIT_COLOR, markeredgecolor=PAGE_BG,
               markersize=9, label="Scaffolding-appropriate situations"),
        Line2D([0], [0], marker="o", linestyle="none",
               markerfacecolor=RIGOR_SIT_COLOR, markeredgecolor=PAGE_BG,
               markersize=9, label="Rigor-appropriate situations"),
        Line2D([0], [0], color=GOOD_COLOR, lw=3, alpha=0.6,
               label="Increased usage in appropriate situations"),
        Line2D([0], [0], color=BAD_COLOR, lw=3, alpha=0.6,
               label="Decreased usage in appropriate situations"),
        Line2D([0], [0], color=NEUTRAL_BAR, lw=3, alpha=0.6,
               label="Neutral tactic"),
    ]
    plt.tight_layout(rect=[0, 1.0 / figh, 1, 1])
    fig.legend(handles=handles, loc="lower center", ncol=3,
               fontsize=10, frameon=False,
               bbox_to_anchor=(0.5, 0.25 / figh))
    fig.savefig(out_pdf, dpi=150, bbox_inches="tight")
    print(f"wrote {out_pdf.relative_to(REPO)}")
    return fig


In [ ]:
fig_s = plot_orientation_grid("scaffolding", OUT_DIR / "figure_dumbbell_scaffolding.pdf")
fig_r = plot_orientation_grid("rigor",       OUT_DIR / "figure_dumbbell_rigor.pdf")
fig_n = plot_orientation_grid("neutral",     OUT_DIR / "figure_dumbbell_neutral.pdf")
fig_s
